In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 2.1 Lagrangian Mechanics with SymPy

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume II — Analytical Mechanics",
    number="2.1",
    title="Lagrangian Mechanics with SymPy",
    blurb="From a single scalar — the Lagrangian — to the equations of "
    "motion of any system, derived symbolically and integrated "
    "numerically. The workhorse tool for the rest of the series.",
    difficulty="intermediate",
    estimate="75–100 min",
)

## Notebook overview

Volume I derived equations of motion by hand and from Newton's laws. That works
for a projectile or a single pendulum, but the algebra grows vicious fast: the
double pendulum of [§1.3](../01-elementary-mechanics/double-pendulum.ipynb) took a page of trigonometry, and we only trusted
the result because we *cross-checked it against SymPy*. Analytical mechanics
turns that relationship around. Instead of summing forces and constraint
tensions vector by vector, we write down **one scalar** (the Lagrangian
$\mathcal L = T - V$) and a single mechanical recipe, the Euler–Lagrange
equations, produces the equations of motion for *every* coordinate at once.

The recipe is pure calculus on a scalar, which means a computer algebra system
can carry it out mechanically and without error. This notebook builds that tool
(a small `euler_lagrange` engine and a `to_numeric` bridge to `solve_ivp`) and
then puts it to work on a sequence of systems: the harmonic oscillator and the
pendulum (to trust the engine against answers we know), the Atwood machine (to
see a constraint vanish), the double pendulum (where the symbolic derivation
*becomes* the primary method, with the hand algebra of [§1.3](../01-elementary-mechanics/double-pendulum.ipynb) now the thing being
checked), a cart-pole (painful by hand, lovely in motion), and a central-force
orbit (where a cyclic coordinate hands us a conservation law for free). This
engine is the workhorse we will reuse throughout Volume II and beyond.

> **How to read the checks.** Each exercise ends with a validation that
> compares our result to an expected physical fact. A ✗ does *not* by itself
> mean the answer is wrong: it means the output didn't match what the check
> expected, which may be a real error, a different-but-valid convention (a
> sign, a unit, an array order), or simply too tight a tolerance. Treat a ✗ as
> a prompt to locate the discrepancy; passing is strong evidence of
> correctness, not proof.

> **Scope.** This is a working review, not a textbook chapter. For the
> variational foundations (Hamilton's principle, the calculus of variations,
> the full derivation of the Euler–Lagrange equations), see Nolting,
> *Theoretical Physics 2* {cite}`nolting2`, and Goldstein, Poole & Safko,
> *Classical Mechanics* {cite}`goldstein`.

## Theory in brief

### The action and Hamilton's principle

A mechanical system described by generalised coordinates $q_i(t)$ has a
**Lagrangian** $\mathcal L(q_i, \dot q_i, t)$, and the **action** is its time
integral along a path,

```{math}
:label: eq-action
S[q] = \int_{t_1}^{t_2} \mathcal L\bigl(q_i, \dot q_i, t\bigr)\, \mathrm dt .
```

**Hamilton's principle** states that the path actually taken between two fixed
endpoints is a *stationary point* of the action: $\delta S = 0$. This is a
remarkable repackaging of mechanics: the whole of the dynamics is encoded in a
single number $S$ being extremal, with no mention of forces. The variational
derivation (deferred to {cite}`nolting2,goldstein`) turns $\delta S = 0$ into a
differential equation for each coordinate.

### The Euler–Lagrange equations

Requiring $\delta S = 0$ for arbitrary variations vanishing at the endpoints
yields, for each generalised coordinate $q_i$, the **Euler–Lagrange equation**

```{math}
:label: eq-el
\frac{\mathrm d}{\mathrm dt}\!\left(\frac{\partial \mathcal L}{\partial \dot q_i}\right)
  - \frac{\partial \mathcal L}{\partial q_i} = 0 .
```

There is one such equation per coordinate, and together they are equivalent to
Newton's laws, but expressed entirely in terms of derivatives of the scalar
$\mathcal L$. No free-body diagrams, no constraint forces to resolve.

### Generalised coordinates and $\mathcal L = T - V$

For a system with kinetic energy $T$ and potential energy $V$, the Lagrangian is
simply their difference,

```{math}
:label: eq-lagrangian
\mathcal L = T - V ,
```

written in whatever **generalised coordinates** best describe the system. This
freedom is the practical heart of the method. A pendulum bob is constrained to a
circle; in Cartesian $(x,y)$ that constraint needs an unknown tension force, but
in the single angle $\theta$ the constraint is *built into the coordinate* and
the tension never appears. Choosing coordinates that automatically satisfy the
constraints removes the constraint forces from the problem entirely.

### Conjugate momenta and cyclic coordinates

The quantity multiplying $\dot q_i$ in the dynamics is the **conjugate (or
generalised) momentum**

```{math}
:label: eq-conjmom
p_i = \frac{\partial \mathcal L}{\partial \dot q_i} .
```

If $\mathcal L$ does not depend on a coordinate $q_i$ explicitly (such a $q_i$
is called **cyclic**, or ignorable) then {eq}`eq-el` collapses to
$\mathrm dp_i/\mathrm dt = 0$: the conjugate momentum is **conserved**. This is
the seed of the deep link between symmetry and conservation (Noether's theorem,
[§2.2](noether.ipynb)): a coordinate the Lagrangian ignores corresponds to a quantity
nature conserves. Translational invariance gives momentum conservation;
rotational invariance (a cyclic angle) gives angular-momentum conservation.

### The computational idea

Equation {eq}`eq-el` is nothing but differentiation of a scalar followed by
solving for the highest derivatives. A computer algebra system performs exactly
those operations exactly: no transcription slips, no dropped terms. So the plan
is: write $\mathcal L$, let SymPy form and solve {eq}`eq-el` for the
accelerations $\ddot q_i$, then *lambdify* those symbolic accelerations into a
numerical right-hand side and integrate with `solve_ivp`. We met this idea in
[§1.3](../01-elementary-mechanics/double-pendulum.ipynb) as a *check* on a hand derivation; here it becomes the primary method.

---
## Setup

We use SymPy for the symbolic derivation and NumPy/SciPy for the numerics. The
two helpers below are the reusable core of the notebook:

- `euler_lagrange(L, coords, t)` forms {eq}`eq-el` for each coordinate and solves
  the coupled set for the second derivatives, returning a dict
  `{q_i'' : expression}`.
- `to_numeric(acc_dict, coords, t, param_values)` lambdifies those accelerations
  and packages them as a first-order right-hand side `f(t, y)` for `solve_ivp`,
  with the interleaved state `y = [q0, q0', q1, q1', …]`.

In [ ]:
import numpy as np
import sympy as sp
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

from ecp import validate
from ecp.animate import show

t = sp.symbols("t")  # the time variable shared by all dynamical coordinates


def euler_lagrange(L, coords, t):
    """Solve the Euler-Lagrange equations of a Lagrangian.

    Forms the Euler-Lagrange equation (eq-el in the theory section) for each
    coordinate and solves the coupled set for the accelerations — the bridge
    from a Lagrangian to integrable equations of motion.

    Parameters
    ----------
    L : sympy.Expr
        The Lagrangian.
    coords : sequence
        Generalised coordinates (Functions of ``t``).
    t : sympy.Symbol
        Time symbol.

    Returns
    -------
    dict
        Mapping each second derivative to its expression.
    """
    eqs = [sp.diff(sp.diff(L, sp.diff(q, t)), t) - sp.diff(L, q) for q in coords]
    accs = [sp.diff(q, t, 2) for q in coords]
    return sp.solve(eqs, accs, dict=True)[0]


def to_numeric(acc_dict, coords, t, param_values):
    """Lambdify Euler-Lagrange accelerations into a `solve_ivp` right-hand side.

    Substitutes parameter values and compiles the symbolic accelerations to a fast numeric ODE function.

    Parameters
    ----------
    acc_dict : dict
        The Euler-Lagrange acceleration map.
    coords : sequence
        Coordinates.
    t : sympy.Symbol
        Time symbol.
    param_values : dict
        Numeric values for the parameters.

    Returns
    -------
    callable
        ``rhs(t, y)`` for `scipy.integrate.solve_ivp`.
    """
    param_syms = list(param_values)
    pvals = [param_values[s] for s in param_syms]
    qdots = [sp.diff(q, t) for q in coords]
    args = list(coords) + qdots + param_syms
    acc_funcs = [sp.lambdify(args, acc_dict[sp.diff(q, t, 2)], "numpy") for q in coords]

    def rhs(_t, y):
        q = y[0::2]
        v = y[1::2]
        call = (*q, *v, *pvals)
        dy = np.empty_like(y)
        dy[0::2] = v
        dy[1::2] = [f(*call) for f in acc_funcs]
        return dy

    return rhs

## Exercise 1 — Build the Euler–Lagrange engine

The whole notebook rests on one function. The Euler–Lagrange equation
{eq}`eq-el` is, per coordinate, $\frac{\mathrm d}{\mathrm dt}(\partial \mathcal
L/\partial \dot q) - \partial \mathcal L/\partial q = 0$; SymPy can form each
piece with `sp.diff` and solve the resulting set for the accelerations. The
`euler_lagrange` helper in Setup does exactly this. The right way to trust a new
symbolic tool is to run it on a system whose answer we already know cold.

Apply the engine to the simple harmonic oscillator, whose Lagrangian
{eq}`eq-lagrangian` is $\mathcal L = \tfrac12 m\dot x^2 - \tfrac12 k x^2$,
and confirm it returns Newton's $\ddot x = -(k/m)\,x$.

1. Define the symbols $m,k$ (`sympy.symbols`) and the coordinate $x(t)$
   (`sympy.Function`), and write $\mathcal L$.
2. Run `euler_lagrange` and read off $\ddot x$; check with `sympy.simplify`
   that it equals $-(k/m)x$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    sp.simplify(sho_acc - (-(k / m) * x)) == 0,
    "the engine recovers the SHO equation of motion ẍ = -(k/m)x",
    f"engine returned ẍ = {sho_acc}",
)

## Exercise 2 — The simple pendulum from its Lagrangian

Now a genuinely constrained system. A pendulum bob is stuck on a circle of
radius $\ell$; in Cartesian coordinates that demands an unknown string tension,
but in the single angle $\theta$ the constraint is automatic and the tension
never enters. With $T = \tfrac12 m\ell^2\dot\theta^2$ and $V = -mg\ell\cos\theta$
(measuring $\theta$ from straight down), the Lagrangian {eq}`eq-lagrangian` is
$\mathcal L = \tfrac12 m\ell^2\dot\theta^2 + mg\ell\cos\theta$.

1. Write $\mathcal L$ in $\theta$ and run the engine; confirm via
   {eq}`eq-el` that $\ddot\theta = -(g/\ell)\sin\theta$, with no tension in
   sight.
2. Lambdify the acceleration (via `to_numeric`) and integrate one
   large-amplitude swing, released from $\theta=2.0$ rad at rest, with
   `scipy.integrate.solve_ivp` (`DOP853`, `rtol=1e-11`, `atol=1e-12`) over a
   dense `t_eval`; plot $\theta(t)$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    sp.simplify(pend_acc + (g / ell) * sp.sin(theta)) == 0,
    "pendulum EOM is θ'' = -(g/l) sinθ",
    f"engine returned θ̈ = {sp.simplify(pend_acc)}",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 3 — A constrained system made easy: the Atwood machine

The Atwood machine (two masses $m_1, m_2$ hanging over a frictionless pulley on
an inextensible string) is the textbook case for *seeing a constraint
disappear*. The Newtonian route introduces the string tension $T$ as an unknown
and needs two equations to eliminate it. The Lagrangian route uses one
generalised coordinate $s$, the length of string on mass $m_1$'s side: if $m_1$
descends by $s$, then $m_2$ rises by $s$, so the inextensible-string constraint
is *already encoded*. The tension never appears.

**Your task.** With both masses moving at speed $\dot s$, the kinetic energy
is $T = \tfrac12(m_1+m_2)\dot s^2$ and the potential is $V = -m_1 g s + m_2 g s$
(heights $\mp s$). Form $\mathcal L = T - V$, run the `euler_lagrange` engine
from Setup, and confirm the classic result
$\ddot s = (m_1 - m_2)g/(m_1+m_2)$.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.check(
    sp.simplify(atwood_acc - (m1 - m2) * g / (m1 + m2)) == 0,
    "Atwood acceleration is (m1-m2)g/(m1+m2)",
    f"engine returned s̈ = {sp.simplify(atwood_acc)}",
)

## Exercise 4 — Symbolic → numeric pipeline; energy as the correctness gate

A symbolic acceleration is only useful once it runs as a number. The
`to_numeric` bridge lambdifies the EL accelerations into a `solve_ivp`
right-hand side; the question is whether the resulting trajectory is *correct*.
For an autonomous conservative system the total energy $E = T + V$ is constant,
so, exactly as in Volume I, the relative drift of $E$ along the integrated
trajectory is an independent gate on the whole pipeline (derivation, lambdify,
and integration together).

**Your task.** Take the pendulum of Exercise 2, reconstruct its energy
$E = \tfrac12 m\ell^2\dot\theta^2 - mg\ell\cos\theta$ as an array along the
trajectory you already integrated, and confirm it is conserved to tight
tolerance.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.conserved(
    E_pend,
    "energy conserved along the integrated EL trajectory",
    rel_drift=1e-6,
)

## Exercise 5 — The double pendulum, derived not hand-coded (cross-check vs. 1.3)

This is the exercise that justifies the whole apparatus. In [§1.3](../01-elementary-mechanics/double-pendulum.ipynb) we
derived the double-pendulum equations of motion by hand: a page of trigonometry
that we only trusted after checking it against SymPy. Here we *invert* that: the
engine derives the EOM straight from the Lagrangian (the hand derivation in
[§1.3](../01-elementary-mechanics/double-pendulum.ipynb)), and the hand-coded formula becomes the thing under test. With the bob positions
$x_1=\ell_1\sin\theta_1,\ y_1=-\ell_1\cos\theta_1$ and $x_2=x_1+\ell_2\sin\theta_2,\
y_2=y_1-\ell_2\cos\theta_2$, the Lagrangian {eq}`eq-lagrangian` is $\mathcal L =
T - V$ with $T=\tfrac12 m_1(\dot x_1^2+\dot y_1^2)+\tfrac12 m_2(\dot x_2^2+\dot
y_2^2)$ and $V = m_1 g y_1 + m_2 g y_2$.

1. Build the double-pendulum Lagrangian and run the engine to get
   $\ddot\theta_1, \ddot\theta_2$ via {eq}`eq-el`.
2. Lambdify both accelerations (`sympy.lambdify`) and compare them to the
   validated hand-coded `deriv` from [§1.3](../01-elementary-mechanics/double-pendulum.ipynb) at several random states
   (a seeded `numpy.random.default_rng`). They should agree to machine
   precision: the symbolic derivation reproduces the hand algebra exactly,
   while being immune to the slips that make hand derivations so error-prone.

In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.close(
    max_diff,
    0.0,
    "SymPy double-pendulum EOM matches the validated 1.3 hand derivation",
    atol=1e-9,
)

## Exercise 6 — A cart-pole, end to end (worked animation)

Now a system where the hand derivation is genuinely unpleasant but the engine
shrugs: the **cart-pole**, a pendulum of mass $m_p$ hanging from a cart of mass
$M$ that slides freely on a horizontal rail. The two coordinates are the cart
position $x$ and the pendulum angle $\theta$ (from straight down); they are
*coupled*: the swinging pole shoves the cart, the moving cart drives the pole.
With the bob at $x_p = x + \ell\sin\theta,\ y_p = -\ell\cos\theta$, the Lagrangian
{eq}`eq-lagrangian` is
$\mathcal L = \tfrac12 M\dot x^2 + \tfrac12 m_p(\dot x_p^2 + \dot y_p^2) + m_p g
\ell\cos\theta$.

Note that $x$ does not appear in $\mathcal L$: it is **cyclic** {eq}`eq-conjmom`,
so the total horizontal momentum is conserved (we will return to exactly this in
Exercise 7). The system is autonomous, so the total energy is conserved too, and
that is what we validate. This is the worked animation; you build the second in
Exercise 7.

The animation shows the cart sliding while the pole swings. Released nearly
inverted, the pole rocks the cart back and forth.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.conserved(
    E_cp,
    "the integrated cart-pole motion conserves total energy (autonomous system)",
    rel_drift=1e-5,
)

## Exercise 7 — Cyclic coordinate ⇒ conserved momentum (student-implemented animation)

The cart-pole hinted at it; here we make it the point. A particle of mass $m$
moving in a central potential $V(r)$ is best described in polar coordinates
$(r,\varphi)$, with kinetic energy $T=\tfrac12 m(\dot r^2 + r^2\dot\varphi^2)$,
so the Lagrangian {eq}`eq-lagrangian` is $\mathcal L = \tfrac12 m(\dot r^2 +
r^2\dot\varphi^2) - V(r)$. The angle $\varphi$ does **not** appear (it is
cyclic), so by {eq}`eq-conjmom` its conjugate momentum
$p_\varphi = \partial\mathcal L/\partial\dot\varphi = m r^2\dot\varphi$ is
conserved. That conserved $p_\varphi$ is the **angular momentum**, and its
constancy is Kepler's second law: the radius sweeps equal areas in equal times.

1. Build the central-potential Lagrangian (use $V(r) = -k/r$), confirm
   $\partial\mathcal L/\partial\varphi = 0$ symbolically (`sympy.diff`), and
   identify $p_\varphi = m r^2\dot\varphi$ from {eq}`eq-conjmom`.
2. Integrate a bound orbit with the engine and `to_numeric` (which lambdifies
   the accelerations into a `scipy.integrate.solve_ivp` right-hand side,
   `DOP853`).
3. **Build the animation** of the orbiting particle with the radius vector
   drawn from the centre, so the equal-area sweep is visible. You have the
   trajectory `(x, y)` below; assemble a `FuncAnimation` showing the moving
   particle, its trail, and the sweeping radial line, `plt.close(fig)`, then
   display with `ecp.animate.show`.

> A ✗ on the final check is about the **`p_phi` time series we computed** from
> the trajectory, not the animation: any correct drawing of the same orbit is
> fine. If it fails, inspect $p_\varphi(t) = m r^2\dot\varphi$ along the
> solution.

In [ ]:
# (solution hidden on the public site)


In [ ]:
# (solution hidden on the public site)


In [ ]:
validate.conserved(
    p_phi,
    "the conjugate momentum to the cyclic coordinate φ is conserved",
    rel_drift=1e-5,
)

## Notebook summary

- A symbolic **Euler–Lagrange engine** built over SymPy (and packaged for the
  rest of the series as `ecp.mechanics.euler_lagrange`): the equations of
  motion read straight off a Lagrangian for the pendulum, the Atwood
  machine, the double pendulum (cross-checked against the hand-coded formula of [§1.3](../01-elementary-mechanics/double-pendulum.ipynb)), and a cart-pole.
- The symbolic → numeric pipeline (`sympy.lambdify` into `scipy.integrate.solve_ivp`),
  with energy conservation as the correctness gate, and a cyclic coordinate yielding a
  conserved momentum.

## Outlook

- **Velocity-dependent potentials.** A charged particle in a magnetic field has
  a Lagrangian with a $\dot{\mathbf q}\cdot\mathbf A$ term; the same engine
  handles it and yields the Lorentz force: a forward link to Volume III.
- **When the constraint force itself is wanted.** Lagrange multipliers reintroduce the
  constraint (and its force) explicitly: useful when the tension or normal force
  is the thing one cares about.
- **Dissipation.** A Rayleigh dissipation function extends {eq}`eq-el` to
  friction and drag, recovering the damped systems of Volume I.
- **Toward Hamiltonian mechanics.** Legendre-transforming $\mathcal L$ in the
  velocities gives the Hamiltonian $H(q,p)$ and Hamilton's equations: the
  subject of [§2.3](hamiltonian-phase-flow.ipynb), and the natural home of the conjugate momenta
  {eq}`eq-conjmom` introduced here.

### References

```{bibliography}
:filter: docname in docnames
```

In [ ]:
from ecp.style import footer

footer()